# Lab 9: OpenAlex LSA and Clustering

**COMPSS 211 | Fall 2026 | Student copy**

Create lightweight document embeddings and inspect clusters after the embeddings lecture.

**Date/deadline:** Friday, November 6, 2026

Work through each task and replace the response placeholders with your own answers.

## Scenario

A teammate has reduced a small set of text features with LSA and clustered the resulting document vectors. Reproduce the steps, read the source abstracts, and decide whether any cluster label is defensible.

## Goal

- Apply TruncatedSVD (LSA).
- Cluster with fixed randomness.
- Inspect abstracts before naming clusters.

## Keep handy

- **Required:** LSA and KMeans.
- **Optional:** A guarded Word2Vec extension may be attempted in a separate environment.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## Practice: LSA embedding and cautious cluster labels

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
works = pd.read_csv(course_data_path("openalex_berkeley_abstracts_2024_sample.csv")).dropna(subset=["abstract"])
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=1500)
matrix = vectorizer.fit_transform(works["abstract"])
lsa = TruncatedSVD(n_components=8, random_state=211)
embeddings = lsa.fit_transform(matrix)
clusterer = KMeans(n_clusters=5, random_state=211, n_init=30)
works["cluster"] = clusterer.fit_predict(embeddings)
display(works.groupby("cluster").agg(
    documents=("openalex_id", "count"),
    example_domain=("primary_domain", lambda values: values.mode().iat[0]),
))

### Optional extension

Word2Vec may be explored in a separate environment. It is not part
of the pinned course dependencies and is not required for completion.

### Your notes

Before you leave, write down one thing you can now do and one question you still have.

> Write your notes here.

## Exit

The last 10 minutes are reserved for the three-question quiz on Monday's material. Use the lab to practice the ideas before you answer from memory.